# Person 3 — International trade indicators
Research question: To what extent can economic indicators predict future price instability in critical minerals used in semiconductor manufacturing?

Run all cells in order. This notebook uses only the Python standard library (no pandas installation required). Open it in VS Code with the Jupyter extension and select a Python kernel. Keep the `input` folder beside the notebook. Outputs are written to `output`; original files are unchanged.

## What you must do
1. Audit the coverage before calculating indicators.
2. Clean Comtrade values and retain classification and quality flags.
3. Calculate imports, exports, and trade balance (exports minus imports) only for matching observations. Missing flows remain missing.
4. Clean USGS import reliance without converting inequalities to exact numbers.
5. Calculate USGS supplier concentration bounds, preserving the 2021–2024 period.
6. Export separate tables; do not force incompatible product definitions into a merged model dataset.

**Current limitations:** Comtrade contains only 2025, chapter 26, partner World. It cannot supply historical mineral-specific bilateral HHI. USGS reliance refers to U.S. 2025, while supplier shares cover 2021–2024. Neither should be copied backwards across historical years. This notebook prepares descriptive indicators, not a forecasting-ready dataset.


In [1]:
from pathlib import Path
import csv, re, math
from collections import Counter, defaultdict

# Run with the notebook directory as your working directory, or edit BASE.
BASE = Path.cwd()
if not (BASE / 'input').exists() and (BASE / 'trade_indicators' / 'input').exists():
    BASE = BASE / 'trade_indicators'
INPUT, OUTPUT = BASE / 'input', BASE / 'output'
OUTPUT.mkdir(exist_ok=True)
FILES = {
    'trade': 'UNComtrade-trade-statistics-official-CAHS-20261121-1055.csv',
    'reliance': 'MCS2026_Fig2_Net_Import_Reliance.csv',
    'sources': 'MCS2026_Fig3_Major_Import_Sources.csv',
}
def read_csv(name):
    with (INPUT / FILES[name]).open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))
def write_csv(name, rows, fields):
    with (OUTPUT / name).open('w', encoding='utf-8', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
        w.writeheader(); w.writerows(rows)
def number(s):
    s = str(s).strip().replace(',', '')
    if s in ('', 'NA', 'N/A', 'W', '—', 'XX'):
        return None
    try:
        v = float(s)
        return v if math.isfinite(v) else None
    except ValueError:
        return None
def unique_rows(rows):
    seen, result = set(), []
    for r in rows:
        key = tuple(sorted(r.items()))
        if key not in seen:
            seen.add(key); result.append(r)
    return result
def normalized(s):
    # Normalize spelling/case only, not chemical form or commodity scope.
    return re.sub(r'\s+', ' ', s.strip()).casefold()
trade_raw, reliance_raw, sources_raw = [read_csv(k) for k in FILES]
for name, rows in zip(FILES, (trade_raw, reliance_raw, sources_raw)):
    print(name, 'rows:', len(rows))


trade rows: 241
reliance rows: 68
sources rows: 191


## 1. Coverage audit
Check years, products, partners and classification revisions. Chapter 26 is a broad product group, not gallium or a semiconductor-mineral basket. Reporters are countries reporting trade; these are not all U.S. observations. Different HS revisions must remain identifiable.


In [2]:
audit = []
for field in ('refYear', 'cmdCode', 'cmdDesc', 'partnerDesc', 'flowCode', 'classificationCode'):
    values = sorted({r[field] for r in trade_raw})
    audit.append({'field':field, 'values':' | '.join(values)})
    print(field, values)
historical = len({r['refYear'] for r in trade_raw}) > 1
bilateral = any(r['partnerCode'] != '0' for r in trade_raw)
mineral_detail = any(len(r['cmdCode']) >= 6 for r in trade_raw)
print('Historical:', historical, '| Bilateral partners:', bilateral, '| Detailed codes:', mineral_detail)
write_csv('coverage_audit.csv', audit, ['field','values'])


refYear ['2025']
cmdCode ['26']
cmdDesc ['Ores, slag and ash']
partnerDesc ['World']
flowCode ['M', 'X']
classificationCode ['H3', 'H5', 'H6']
Historical: False | Bilateral partners: False | Detailed codes: False


## 2. Clean Comtrade
Retain reported trade value (`primaryValue`, USD). Do not add CIF and FOB values together. Retain zero/missing weight as raw metadata; here zero weights are not usable physical quantities because all downloaded records are broad aggregates with no valid quantity unit. Export balance is a value-based measure; import and export valuation bases may differ.
Only total customs, transport and secondary-partner dimensions are selected to avoid mixing breakdowns and totals. Conflicting duplicate analytical keys cause an error rather than silent summation.


In [3]:
trade = []
for r in unique_rows(trade_raw):
    if not (r['freqCode']=='A' and r['flowCode'] in ('M','X')
            and r['partner2Code']=='0' and r['customsCode']=='C00'
            and r['motCode']=='0' and r['mosCode']=='0'):
        continue
    value = number(r['primaryValue'])
    if value is not None and value < 0:
        raise ValueError('Negative trade value: investigate source record')
    weight = number(r['netWgt'])
    trade.append({
        'year':int(r['refYear']), 'reporter_code':r['reporterCode'],
        'reporter_iso':r['reporterISO'], 'reporter_country':r['reporterDesc'],
        'partner_code':r['partnerCode'], 'partner_country':r['partnerDesc'],
        'classification':r['classificationCode'], 'commodity_code':r['cmdCode'],
        'commodity_description':r['cmdDesc'], 'flow':r['flowCode'],
        'trade_value_usd':value, 'net_weight_raw':r['netWgt'],
        'net_weight_tonnes':weight/1000 if weight is not None and weight>0 else None,
        'weight_estimated':r['isNetWgtEstimated'], 'is_reported':r['isReported'],
        'is_aggregate':r['isAggregate'], 'is_original_classification':r['isOriginalClassification'],
        'legacy_estimation_flag':r['legacyEstimationFlag'],
        'cif_value_usd':number(r['cifvalue']), 'fob_value_usd':number(r['fobvalue']),
    })
key_fields = ['year','reporter_code','partner_code','classification','commodity_code','flow']
keys = [tuple(r[k] for k in key_fields) for r in trade]
assert len(keys)==len(set(keys)), 'Conflicting duplicate keys: investigate before aggregation'
write_csv('comtrade_clean.csv', trade, list(trade[0]))
print('Clean records:',len(trade))


Clean records: 241


## 3. Imports, exports and trade balance
Use World totals once, not alongside their component partners. Missing imports/exports are not zero. The net-import share of total trade below is a descriptive trade ratio, **not** USGS net import reliance (which relates to apparent consumption).


In [4]:
groups = defaultdict(dict)
for r in trade:
    if r['partner_code']=='0':
        key = tuple(r[k] for k in ['year','reporter_code','classification','commodity_code'])
        groups[key][r['flow']] = r
balances = []
for key, flows in groups.items():
    sample = next(iter(flows.values()))
    m, x = flows.get('M',{}).get('trade_value_usd'), flows.get('X',{}).get('trade_value_usd')
    complete = m is not None and x is not None
    total = m+x if complete else None
    balances.append({
        'year':key[0], 'reporter_code':key[1], 'reporter_country':sample['reporter_country'],
        'classification':key[2], 'commodity_code':key[3],
        'commodity_description':sample['commodity_description'],
        'imports_usd':m, 'exports_usd':x,
        'trade_balance_usd':x-m if complete else None,
        'net_imports_usd':m-x if complete else None,
        'net_import_share_total_trade':(m-x)/total if complete and total>0 else None,
        'both_flows_available':complete,
    })
write_csv('trade_balances.csv', balances, list(balances[0]))
print('Country-product-year balances:',len(balances))
print('Missing a flow:',sum(not r['both_flows_available'] for r in balances))
print('Example:',balances[0])


Country-product-year balances: 126
Missing a flow: 11
Example: {'year': 2025, 'reporter_code': '20', 'reporter_country': 'Andorra', 'classification': 'H6', 'commodity_code': '26', 'commodity_description': 'Ores, slag and ash', 'imports_usd': 1120.705, 'exports_usd': 6956.918, 'trade_balance_usd': 5836.213, 'net_imports_usd': -5836.213, 'net_import_share_total_trade': -0.7225161411964881, 'both_flows_available': True}


## 4. Import dependency: preserve inequalities
Use Fig2 as the authoritative reliance field for this output. Preserve `>50`, `<25` etc. as censored observations. Bounds below describe intervals; they are not exact imputed estimates. Fig3 NIR is retained later for comparison, because it may simplify inequalities or differ from Fig2.


In [5]:
def parse_reliance(raw):
    raw = raw.strip()
    match = re.fullmatch(r'([<>]?)\s*(\d+(?:\.\d+)?)',raw)
    if not match:
        return None, None, None, 'missing_or_unparsed'
    op, val = match.group(1), float(match.group(2))
    if not 0 <= val <= 100:
        raise ValueError(raw)
    if op=='>': return None, val, 100.0, 'greater_than'
    if op=='<': return None, 0.0, val, 'less_than'
    return val, val, val, 'exact'
reliance = []
for r in unique_rows(reliance_raw):
    exact, low, high, relation = parse_reliance(r['Net_Import_Reliance_pct_2025'])
    reliance.append({
        'commodity':r['Commodity'].strip(), 'commodity_key':normalized(r['Commodity']),
        'reporter_country':'United States', 'reference_year':2025,
        'reliance_raw':r['Net_Import_Reliance_pct_2025'],
        'reliance_exact_pct':exact, 'reliance_lower_bound_pct':low,
        'reliance_upper_bound_pct':high, 'reliance_relation':relation,
        'major_sources_2021_2024':r['Major_Import_Sources_2021_24'],
        'notes':r['Notes'], 'source':r['Source'],
    })
write_csv('usgs_import_dependency_2025.csv', reliance, list(reliance[0]))
print('Reliance records:',len(reliance),Counter(r['reliance_relation'] for r in reliance))


Reliance records: 68 Counter({'exact': 53, 'greater_than': 11, 'less_than': 4})


## 5. Supplier shares and concentration bounds (USGS 2021–2024)
For shares in fractions, HHI = sum(share²). Higher concentration means fewer/dominant suppliers, but does not establish causality or prove future instability.

`Other` is not one country. The sum of named shares squared is a conservative lower bound; adding `Other²` gives an upper bound under the assumption that the residual comes from one supplier. The lower bound need not be attainable. Report both, not an exact HHI. Use only groups with valid country shares summing to 100 (within 0.01 percentage points). NA/unknown supplier rows cannot establish HHI. Rounded reported shares introduce additional uncertainty not captured by these bounds.

USGS says some Hong Kong flows are included with China; retain this grouping. These are U.S. import-source shares, not worldwide production shares.


In [6]:
source_rows, by_mineral = [], defaultdict(list)
for r in unique_rows(sources_raw):
    item = {
        'commodity':r['Commodity'].strip(), 'commodity_key':normalized(r['Commodity']),
        'reporter_country':'United States','period_start':2021,'period_end':2024,
        'supplier_country':r['Import_Source_2021_24'].strip(),
        'share_pct':number(r['Percent']), 'share_raw':r['Percent'],
        'nir_fig3_raw':r['NIR'], 'critical_mineral_fig3_raw':r['Critical_Mineral'],
        'source':r['Source'],
    }
    source_rows.append(item); by_mineral[item['commodity_key']].append(item)
concentration = []
for mineral, records in by_mineral.items():
    names = [r['supplier_country'] for r in records]
    assert len(names)==len(set(names)), 'Duplicate supplier within commodity'
    valid_shares = all(r['share_pct'] is not None and 0<=r['share_pct']<=100 for r in records)
    known_names = all(normalized(r['supplier_country']) not in ('na','n/a','unknown','') for r in records)
    total = sum(r['share_pct'] for r in records) if valid_shares else None
    valid = valid_shares and known_names and abs(total-100)<0.01
    named = [r for r in records if normalized(r['supplier_country'])!='other']
    residual = sum(r['share_pct'] for r in records if normalized(r['supplier_country'])=='other') if valid else None
    low = sum((r['share_pct']/100)**2 for r in named) if valid else None
    high = low+(residual/100)**2 if valid else None
    concentration.append({
        'commodity':records[0]['commodity'], 'commodity_key':mineral,
        'reporter_country':'United States','period_start':2021,'period_end':2024,
        'reported_share_sum_pct':total, 'valid_for_hhi_bounds':valid,
        'named_supplier_count':len(named) if known_names else None,
        'largest_named_supplier_share_pct':max(r['share_pct'] for r in named) if valid and named else None,
        'other_share_pct':residual, 'hhi_lower_bound':low, 'hhi_upper_bound':high,
        'hhi_exact':low if valid and residual==0 else None,
        'nir_fig3_raw':records[0]['nir_fig3_raw'],
        'critical_mineral_fig3_raw':records[0]['critical_mineral_fig3_raw'],
    })
write_csv('usgs_supplier_shares_2021_2024.csv',source_rows,list(source_rows[0]))
write_csv('usgs_concentration_bounds_2021_2024.csv',concentration,list(concentration[0]))
print('Commodity groups:',len(concentration))
print('Valid concentration bounds:',sum(r['valid_for_hhi_bounds'] for r in concentration))
print('Gallium:',next(r for r in concentration if r['commodity_key']=='gallium'))


Commodity groups: 47
Valid concentration bounds: 40
Gallium: {'commodity': 'Gallium', 'commodity_key': 'gallium', 'reporter_country': 'United States', 'period_start': 2021, 'period_end': 2024, 'reported_share_sum_pct': 100.0, 'valid_for_hhi_bounds': True, 'named_supplier_count': 4, 'largest_named_supplier_share_pct': 28.0, 'other_share_pct': 16.0, 'hhi_lower_bound': 0.18480000000000002, 'hhi_upper_bound': 0.21040000000000003, 'hhi_exact': None, 'nir_fig3_raw': '100', 'critical_mineral_fig3_raw': 'Y'}


## 6. Commodity mapping for manual review
Do not strip qualifiers such as refined, metal, compounds or silicon/ferrosilicon. The two USGS tables do not always describe identical product scopes. Exact normalized text matches are only provisional candidates; no automatic merged model table is created. Verify critical-mineral classification separately because Fig3's flag should not be treated as authoritative.


In [7]:
fig2_keys = {r['commodity_key']:r for r in reliance}
mapping = []
for r in concentration:
    candidate = fig2_keys.get(r['commodity_key'])
    mapping.append({
        'fig3_commodity':r['commodity'],
        'fig2_exact_text_candidate':candidate['commodity'] if candidate else '',
        'fig2_reliance_raw':candidate['reliance_raw'] if candidate else '',
        'fig3_nir_raw':r['nir_fig3_raw'],
        'review_status':'MANUAL_SCOPE_REVIEW_REQUIRED',
    })
write_csv('commodity_mapping_review.csv',mapping,list(mapping[0]))
print('Provisional exact text matches:',sum(bool(r['fig2_exact_text_candidate']) for r in mapping))


Provisional exact text matches: 30


## 7. Validation and export checks
These checks verify missing-flow handling, reliance inequalities, and concentration arithmetic. No historical HHI is fabricated from World totals.


In [8]:
assert parse_reliance('>50') == (None,50.0,100.0,'greater_than')
assert parse_reliance('NA')[0] is None
assert number('1,187')==1187
for r in balances:
    if r['both_flows_available']:
        assert math.isclose(r['trade_balance_usd'],r['exports_usd']-r['imports_usd'])
    else:
        assert r['trade_balance_usd'] is None
for r in concentration:
    if r['valid_for_hhi_bounds']:
        assert 0<=r['hhi_lower_bound']<=r['hhi_upper_bound']<=1.000001
gallium = next(r for r in concentration if r['commodity_key']=='gallium')
assert math.isclose(gallium['hhi_lower_bound'],0.1848)
assert math.isclose(gallium['hhi_upper_bound'],0.2104)
print('All checks passed.')
print('Outputs:',*[p.name for p in sorted(OUTPUT.glob('*.csv'))],sep='\n- ')


All checks passed.
Outputs:
- commodity_mapping_review.csv
- comtrade_clean.csv
- coverage_audit.csv
- trade_balances.csv
- usgs_concentration_bounds_2021_2024.csv
- usgs_import_dependency_2025.csv
- usgs_supplier_shares_2021_2024.csv


## 8. What to download next and hand to the team
- Re-download bilateral Comtrade data: selected reporter (e.g. United States), all individual partners, imports and exports, detailed verified mineral codes, multiple historical years. Keep World totals separately for completeness checks. Annual frequency is appropriate for an annual model.
- Some HS6 codes combine minerals. Verify year-specific national tariff lines for gallium/germanium where necessary; do not label all chapter 26 trade as semiconductor mineral trade. Track classification changes.
- Bilateral import HHI: for each reporter × consistent mineral/product × year, aggregate imports by supplier, divide each supplier value by the total of individual supplier values, square shares and sum. Exclude World/regional aggregates. Use quantity HHI only when units and completeness permit; compare against World totals to assess coverage.
- Final historical handoff columns: year, reporter, mineral/product definition, code/revision, imports_usd, exports_usd, trade_balance_usd, import_dependency (with definition), supplier_hhi_value, supplier_hhi_quantity where valid, largest_supplier_share, supplier_count, and quality flags.
- Current USGS tables are context snapshots. Never duplicate 2021–2024 supplier shares across individual years, or use 2025 reliance to forecast past outcomes. Do not join U.S. dependence to every reporting country.
- A forecasting model also needs a longer aligned price history, economic indicators and release dates. Lag features to information actually available before the target period. Supplier concentration is a predictive hypothesis, not an established causal result.

Sources: user-supplied CSV exports. Definitions should be checked against [UN Comtrade documentation](https://uncomtrade.org/docs/) and the original USGS MCS2026 notes. No online source was required to inspect these local files.
